# Summary Metrics

Total number of flights in 2022

In [0]:
%sql
SELECT 
    COUNT(*) AS total_fligts 
FROM airlines_lakehouse_2022.gold.fact_flights;

total_fligts
6954636


Average departure delay in minutes in 2022

In [0]:
%sql
SELECT 
    AVG(ABS(dep_delay)) AS avergae_departure_delay 
FROM airlines_lakehouse_2022.gold.fact_flights

avergae_departure_delay
18.03143615280512


Total number of cancelled flights in 2022

In [0]:
%sql
SELECT 
    COUNT(cancellation_key) total_cancelled_flights 
FROM airlines_lakehouse_2022.gold.fact_flights 
WHERE cancellation_key !=0;

total_cancelled_flights
147830


# Airport Performance

Number of flights per month in 2022

In [0]:
%sql
SELECT
    d.month,
    COUNT(*) AS num_flights
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_date d
    ON f.date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;


month,num_flights
1,551993
2,513579
3,587033
4,576982
5,599519
6,597001
7,615206
8,609325
9,577146
10,593008


Number of cancelled flights per month in 2022

In [0]:
%sql
SELECT
    d.month,
    COUNT(*) AS num_flights
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_date d
    ON f.date_key = d.date_key
WHERE f.cancellation_key !=0
GROUP BY d.month
ORDER BY d.month;

month,num_flights
1,25099
2,17928
3,7080
4,11481
5,10143
6,15165
7,9501
8,13041
9,6728
10,3570


Average delay of flights per month in 2022

In [0]:
%sql
SELECT
    d.month,
    ROUND(AVG(ABS(f.dep_delay)),2) AS average_departure_delay 
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_date d
    ON f.date_key = d.date_key
GROUP BY d.month
ORDER BY d.month;

month,average_departure_delay
1,17.3
2,16.96
3,17.92
4,18.48
5,17.73
6,20.37
7,20.13
8,19.37
9,14.84
10,14.46


Top 10 Busiest destination

In [0]:
%sql
SELECT
    a.display_airport_name,
    COUNT(*)
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_airports a
    ON f.destination_airport_key = a.airport_key
GROUP BY a.display_airport_name
ORDER BY COUNT(*) DESC
LIMIT 10;

display_airport_name,COUNT(*)
Hartsfield-Jackson Atlanta International,316767
Chicago O'Hare International,292820
Denver International,275361
Dallas/Fort Worth International,275159
Charlotte Douglas International,213680
Los Angeles International,190459
Harry Reid International,174235
Seattle/Tacoma International,173475
LaGuardia,169512
Phoenix Sky Harbor International,164334


Top 10 average departure delay per destination airport


In [0]:
%sql
SELECT
    a.display_airport_name,
    ROUND(AVG(ABS(f.dep_delay)),2) AS average_departure_delay 
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_airports a
    ON f.origin_airport_key = a.airport_key
GROUP BY a.display_airport_name
ORDER BY average_departure_delay DESC
LIMIT 10;

display_airport_name,average_departure_delay
Pago Pago International,54.53
Watertown Regional,49.0
Elko Regional,39.58
New Castle,38.76
Bishop Airport,37.91
Shenandoah Valley Regional,37.18
Stockton Metro,37.14
Santa Maria Public/Capt. G. Allan Hancock Field,36.41
Greenbrier Valley,34.36
John Murtha Johnstown-Cambria County,34.02


# Carrier Metrics

Total number of flight per carrier in 2022

In [0]:
%sql
SELECT
    c.airline_name,
    COUNT(*) AS num_flights
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_airlines c
ON f.mkt_unique_carrier_key = c.carrier_key
GROUP BY c.airline_name
ORDER BY num_flights DESC;

airline_name,num_flights
American Airlines Inc.,1750416
Delta Air Lines Inc.,1445839
Southwest Airlines Co.,1294853
United Air Lines Inc.,1240798
Alaska Airlines Inc.,381584
JetBlue Airways,272081
Spirit Air Lines,232005
Frontier Airlines Inc.,150867
Allegiant Air,113080
Hawaiian Airlines Inc.,73113


Running total number of flights per carrier per month

In [0]:
%sql
WITH carrier_flight_num AS 
(SELECT
    c.airline_name,
    d.month,
    COUNT(*) AS num_flights
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_date d
JOIN airlines_lakehouse_2022.gold.dim_airlines c
ON f.date_key = d.date_key
AND f.mkt_unique_carrier_key = c.carrier_key
GROUP BY c.airline_name, d.month
ORDER BY d.month)
SELECT
    airline_name,
    month,
    SUM(num_flights) OVER (PARTITION BY airline_name ORDER BY month)
    AS monthly_flights
FROM carrier_flight_num;

airline_name,month,monthly_flights
Alaska Airlines Inc.,1,29536
Alaska Airlines Inc.,2,56567
Alaska Airlines Inc.,3,87904
Alaska Airlines Inc.,4,120311
Alaska Airlines Inc.,5,153955
Alaska Airlines Inc.,6,187610
Alaska Airlines Inc.,7,222761
Alaska Airlines Inc.,8,257649
Alaska Airlines Inc.,9,291189
Alaska Airlines Inc.,10,321948


Running total delay of carriers per month

In [0]:
%sql

WITH carrier_delay AS 
(SELECT
    c.airline_name,
    d.month,
    SUM(ABS(f.dep_delay)) AS total_delay
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_date d
JOIN airlines_lakehouse_2022.gold.dim_airlines c
ON f.date_key = d.date_key
AND f.mkt_unique_carrier_key = c.carrier_key
GROUP BY c.airline_name, d.month
ORDER BY d.month)
SELECT
    airline_name,
    month,
    SUM(total_delay) OVER (PARTITION BY airline_name ORDER BY month)
    AS monyhly_delay
FROM carrier_delay



airline_name,month,monyhly_delay
Alaska Airlines Inc.,1,474677
Alaska Airlines Inc.,2,793796
Alaska Airlines Inc.,3,1172185
Alaska Airlines Inc.,4,1619667
Alaska Airlines Inc.,5,2045663
Alaska Airlines Inc.,6,2501731
Alaska Airlines Inc.,7,2939990
Alaska Airlines Inc.,8,3384664
Alaska Airlines Inc.,9,3807845
Alaska Airlines Inc.,10,4144144


# Aircraft Metrics


Top 10 busiest aircraft

In [0]:
%sql
SELECT 
    a.tail_number_key,
    a.aircraft_manufacturer,
    COUNT(*) AS num_flights
FROM airlines_lakehouse_2022.gold.fact_flights f
JOIN airlines_lakehouse_2022.gold.dim_aircrafts a
ON f.tail_number_key = a.tail_number_key
GROUP BY a.tail_number_key, a.aircraft_manufacturer
ORDER BY num_flights DESC
LIMIT 10;


tail_number_key,aircraft_manufacturer,num_flights
N475HA,Boeing,3033
N492HA,Boeing,3032
N483HA,Boeing,2825
N484HA,Boeing,2813
N476HA,Boeing,2794
N490HA,Boeing,2792
N493HA,Boeing,2784
N478HA,Boeing,2773
N489HA,Boeing,2731
N494HA,Boeing,2553


Aircraft age calculation and classification

In [0]:
%sql
SELECT
    tail_number_key,
    aircraft_manufacturer,
    (2022 - aircraft_year_of_manufacture) as aircraft_age,
    CASE
        WHEN aircraft_age <= 5 THEN "New"
        WHEN aircraft_age > 5 and aircraft_age <= 15 THEN "Mid-age"
        WHEN aircraft_age > 15 and aircraft_age <= 25 THEN "Older"
        ELSE "Very Old"
    END AS aircraft_age_group
FROM airlines_lakehouse_2022.gold.dim_aircrafts
ORDER BY aircraft_age DESC;


tail_number_key,aircraft_manufacturer,aircraft_age,aircraft_age_group
N658DL,Boeing,32,Very Old
N659DL,Boeing,32,Very Old
N309US,Airbus,32,Very Old
N171DN,Boeing,32,Very Old
N660DL,Boeing,32,Very Old
N175DN,Boeing,32,Very Old
N312US,Airbus,32,Very Old
N174DN,Boeing,32,Very Old
N172DN,Boeing,32,Very Old
N641UA,Boeing,31,Very Old
